# PoinTr + PointNet — part-aware colored completion (Colab, tek PyTorch ortam)

Tek notebook: PoinTr fork'unu kurar (clone → deps → CUDA extensions → pointnet2_ops shim →
checkpoint), sonra part-aware colored completion'ı çalıştırır — **dondurulmuş PoinTr** geometriyi
tamamlar, **PyTorch PointNet** parçaları segmentler, her nokta **parçasının rengiyle** boyanır.
Color-head / uçtan-uca 6D yaklaşımı YOK (geometriyi çökertiyordu).

**Kullanım:** Runtime = GPU (Runtime ▸ Change runtime type ▸ GPU). Baştan sona çalıştır.

In [ ]:
!nvidia-smi -L
# GPU görünmüyorsa: Runtime > Change runtime type > Hardware accelerator = GPU

### 1) Python bağımlılıkları (numpy<2 sabit; open3d/timm güncel)

In [ ]:
# torch 2.11 numpy 2.x'e karşı derli -> numpy'yi DOWNGRADE ETME (mixed-install -> mtrand ABI hatası).
# Pinli eski open3d==0.9 / timm==0.4.5 py3.12'de derlenmez; güncelleri kurulur.
!pip install -q easydict h5py matplotlib opencv-python pyyaml scipy \
    tensorboardX tqdm transforms3d einops timm open3d gdown
import numpy as np; print("deps OK | numpy", np.__version__, "(2.x olmalı)")

> ⚠️ **numpy'yi 2.x'te bırak** (torch 2.11 onu ister). Eğer `numpy.dtype size changed` /
> `mtrand` hatası alırsan numpy karışmış demektir → şunu çalıştır ve **Runtime ▸ Restart**:
> `!pip install --force-reinstall --no-cache-dir "numpy==2.0.2"`  — sonra baştan çalıştır.

### 2) Fork'u klonla → `/content/PoinTr`

In [ ]:
import os
os.chdir("/content")
if not os.path.isdir("/content/PoinTr"):
    !git clone https://github.com/eylulpelinkilic/Pelin_Efe_PoinTr.git /content/PoinTr
%cd /content/PoinTr
!git rev-parse --short HEAD

### 3) CUDA build ortamı (sabit değil — otomatik tespit + doğru GPU arch)

In [ ]:
import os, glob, torch
# Colab'da aktif toolkit /usr/local/cuda sembolik linkidir; sürümü hardcode ETME
cuda_home = "/usr/local/cuda" if os.path.isdir("/usr/local/cuda") else sorted(glob.glob("/usr/local/cuda*"))[-1]
os.environ["CUDA_HOME"] = cuda_home
os.environ["PATH"] = f"{cuda_home}/bin:" + os.environ["PATH"]
# eklentiler DOĞRU GPU mimarisi için derlensin (T4=7.5, V100=7.0, A100=8.0, L4=8.9) -> "no kernel image" hatasını önler
cap = torch.cuda.get_device_capability(0)
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{cap[0]}.{cap[1]}"
print("torch", torch.__version__, "| torch-cuda", torch.version.cuda,
      "| CUDA_HOME", cuda_home, "| arch", os.environ["TORCH_CUDA_ARCH_LIST"])
!nvcc --version | tail -2

### 4) `pointnet2_ops` (saf-PyTorch shim — derleme yok)
PoinTr'ın `fps`/`three_nn` gibi ops'ları buna bağlı. Çok yeni torch'ta eski CUDA
repo'su derlenmediği için, kullanılan 6 fonksiyonu saf torch'la enjekte ediyoruz.

In [ ]:
# pointnet2_ops'u DERLEMEK yerine saf-PyTorch SHIM olarak enjekte ediyoruz.
# torch 2.11+cu128 gibi çok yeni stack'te eski CUDA repo'su derlenmiyor. PoinTr sadece
# şu 6 fonksiyonu kullanıyor; hepsi saf torch'la doğru (yerelde brute-force'a karşı test edildi).
# FPS saf-torch döngüsü biraz yavaş ama bu ölçekte (~8k nokta) sorun değil.
import sys, types, torch

def furthest_point_sample(xyz, npoint):        # xyz (B,N,3) -> idx (B,npoint) int32
    B, N, _ = xyz.shape; dev = xyz.device
    idx = torch.zeros(B, npoint, dtype=torch.long, device=dev)
    dist = torch.full((B, N), 1e10, device=dev, dtype=xyz.dtype)
    far = torch.zeros(B, dtype=torch.long, device=dev); ar = torch.arange(B, device=dev)
    for i in range(npoint):
        idx[:, i] = far
        dist = torch.minimum(dist, ((xyz - xyz[ar, far].unsqueeze(1)) ** 2).sum(-1))
        far = torch.max(dist, dim=1).indices
    return idx.int()

def gather_operation(features, idx):           # (B,C,N),(B,S) -> (B,C,S)
    B, C, N = features.shape; idx = idx.long()
    return torch.gather(features, 2, idx.unsqueeze(1).expand(B, C, idx.shape[1])).contiguous()

def three_nn(query, ref):                      # (B,N,3),(B,M,3) -> dist(B,N,3) öklid, idx(B,N,3)
    d = torch.cdist(query, ref)
    dist, idx = torch.topk(d, 3, dim=-1, largest=False, sorted=True)
    return dist.contiguous(), idx.int().contiguous()

def three_interpolate(features, idx, weight):  # (B,C,M),(B,N,3),(B,N,3) -> (B,C,N)
    B, C, M = features.shape; N = idx.shape[1]; idx = idx.long()
    g = torch.gather(features, 2, idx.reshape(B,1,N*3).expand(B,C,N*3)).reshape(B,C,N,3)
    return (g * weight.unsqueeze(1)).sum(-1).contiguous()

def grouping_operation(features, idx):         # (B,C,N),(B,S,K) -> (B,C,S,K)  (SnowFlakeNet için)
    B, C, N = features.shape; _, S, K = idx.shape; idx = idx.long()
    return torch.gather(features, 2, idx.reshape(B,1,S*K).expand(B,C,S*K)).reshape(B,C,S,K).contiguous()

def ball_query(radius, nsample, xyz, new_xyz): # (r,k,(B,N,3),(B,S,3)) -> idx(B,S,k)  (SnowFlakeNet için)
    B, N, _ = xyz.shape; S = new_xyz.shape[1]
    d = torch.cdist(new_xyz, xyz)
    idx = torch.arange(N, device=xyz.device).view(1,1,N).expand(B,S,N).contiguous()
    idx[d > radius] = N
    idx = idx.sort(dim=-1).values[:, :, :nsample]
    first = idx[:, :, 0:1].clone(); first[first == N] = 0
    idx = torch.where(idx == N, first.expand(-1,-1,nsample), idx)
    return idx.int()

_u = types.ModuleType("pointnet2_ops.pointnet2_utils")
for _f in [furthest_point_sample, gather_operation, three_nn, three_interpolate,
           grouping_operation, ball_query]:
    setattr(_u, _f.__name__, _f)
_p = types.ModuleType("pointnet2_ops"); _p.pointnet2_utils = _u
sys.modules["pointnet2_ops"] = _p
sys.modules["pointnet2_ops.pointnet2_utils"] = _u
from pointnet2_ops import pointnet2_utils
print("pointnet2_ops shim enjekte edildi:",
      [n for n in dir(pointnet2_utils) if not n.startswith("_")])

### 5) CUDA extension'ları derle (`chamfer` zorunlu; gridding/cubic GRNet için)

In [ ]:
import subprocess
EXTS = ["chamfer_dist", "gridding", "gridding_loss", "cubic_feature_sampling"]  # emd PoinTr için gerekmez
for ext in EXTS:
    print(f"── building {ext} ──")
    # --no-build-isolation: bu setup.py'ler de torch.utils.cpp_extension'a bağlı
    r = subprocess.run("pip install -q --no-build-isolation .", shell=True,
                       cwd=f"/content/PoinTr/extensions/{ext}", capture_output=True, text=True)
    ok = r.returncode == 0
    print("   ", "✅ ok" if ok else "‼ FAILED")
    if not ok:
        print(r.stdout[-600:]); print(r.stderr[-1800:])

### 6) Her şey import oluyor mu? (GPU smoke test)

In [ ]:
import os, sys
sys.path.insert(0, "/content/PoinTr"); os.chdir("/content/PoinTr")
import torch, numpy as np
print("numpy", np.__version__, "| torch", torch.__version__)
import chamfer, gridding, gridding_distance, cubic_feature_sampling
from pointnet2_ops import pointnet2_utils
from extensions.chamfer_dist import ChamferDistanceL1
from models.PoinTr import PoinTr, Fold, fps
from models.dgcnn_group import DGCNN_Grouper
from models.Transformer import PCTransformer
# gerçekten GPU'da çalışıyor mu: fps + chamfer
x = torch.rand(1, 1024, 3, device="cuda")
idx = pointnet2_utils.furthest_point_sample(x, 128)
d = ChamferDistanceL1()(x, torch.rand(1, 512, 3, device="cuda"))
print("pointnet2 fps:", tuple(idx.shape), "| chamfer:", float(d))
print("✅ PoinTr environment READY")

### 7) Pretrained checkpoint (ShapeNet55)

In [ ]:
import os, subprocess
CKPT = "/content/PoinTr/ckpts/PoinTr_ShapeNet55.pth"
os.makedirs(os.path.dirname(CKPT), exist_ok=True)
if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 50e6:
    subprocess.run(f"gdown 1WzERLlbSwzGOBybzkjBrApwyVMTG00CJ -O {CKPT}", shell=True, check=True)
print("checkpoint MB:", round(os.path.getsize(CKPT)/1e6, 1), " (>400 olmalı)")

---
## Data: occluded partial + GT yükle

Renkli occluded partial'ları (occluded.ipynb çıktısı) yükle, **GT = partial ∪ missing** kur, cache'le.
Aşağıdaki **P1–P4** (dondurulmuş PoinTr geometri) ve **S1–S3** (PointNet part-seg) bunu kullanır.

In [ ]:
import numpy as np, glob, random, zipfile, os, open3d as o3d
from scipy.spatial import cKDTree

def load_ply(p, n=8192):
    pc = o3d.io.read_point_cloud(p)
    if len(pc.points) > n: pc = pc.farthest_point_down_sample(n)   # uniform FPS -> n
    xyz = np.asarray(pc.points); rgb = np.asarray(pc.colors)
    if len(rgb) != len(xyz): rgb = np.zeros_like(xyz)
    return np.concatenate([xyz, rgb], 1).astype(np.float32)

def pc_norm(gt):                                    # unit-sphere (PoinTr eğitim konvansiyonu)
    x = gt[:, :3] - gt[:, :3].mean(0)
    x = x / (np.linalg.norm(x, axis=1).max() + 1e-9)
    return np.concatenate([x, gt[:, 3:6]], 1).astype(np.float32)

def separate_colored(gt, crop=0.5, seed=0):         # -> partial(6D), mask(True=eksik)
    xyz = gt[:, :3]; N = len(gt); nc = int(round(N * crop))
    c = xyz.mean(0); xyzn = (xyz - c) / (np.linalg.norm(xyz - c, axis=1).max() + 1e-9)
    rng = np.random.default_rng(seed); v = rng.standard_normal(3); v /= np.linalg.norm(v)
    order = np.argsort(np.linalg.norm(xyzn - v[None], axis=1))
    mask = np.zeros(N, bool); mask[order[:nc]] = True
    return gt[order[nc:]], mask

def srgb_to_lab(rgb):
    rgb = np.clip(rgb, 0, 1); lin = np.where(rgb > 0.04045, ((rgb + 0.055) / 1.055) ** 2.4, rgb / 12.92)
    M = np.array([[0.4124,0.3576,0.1805],[0.2126,0.7152,0.0722],[0.0193,0.1192,0.9505]])
    xyz = (lin @ M.T) / np.array([0.95047, 1.0, 1.08883]); d = 6 / 29
    f = np.where(xyz > d ** 3, np.cbrt(xyz), xyz / (3 * d ** 2) + 4 / 29)
    return np.stack([116*f[:,1]-16, 500*(f[:,0]-f[:,1]), 200*(f[:,1]-f[:,2])], 1)
def deltaE(a, b): return np.linalg.norm(srgb_to_lab(a) - srgb_to_lab(b), axis=1)
print("yardımcılar hazır")

### Veri: PartAnnotation zip'inden oku → renklendir → **occlude (notebook'ta)**
İndirme yok. `PART_ZIP`'i kendi zip yoluna ayarla; açar, uçak synset'ini (`02691156`) otomatik bulur.

In [ ]:
# --- Veri: ShapeNet-Part uçak (PartAnnotation: points_label/<PARÇA>/<model>.seg binary maskeler) ---
import numpy as np, torch, glob, os, zipfile

PART_SRC = "/content/02691156.zip"        # zip ya da klasör
EXTRACT  = "/content/_spart"
N_PTS, CROP, N_MODELS, CNOISE = 2048, 0.5, 60, 0.03

def _resolve(src):
    if os.path.isdir(src): return src
    if zipfile.is_zipfile(src):
        if not os.path.isdir(EXTRACT):
            with zipfile.ZipFile(src) as z: z.extractall(EXTRACT)
        return EXTRACT
    raise FileNotFoundError(f"'{src}' ne klasör ne geçerli zip. /content: {os.listdir('/content')[:20]}")

ROOT = _resolve(PART_SRC)
hit = glob.glob(f"{ROOT}/**/points/*.pts", recursive=True)
assert hit, f"points/*.pts yok {ROOT}"
PART_DIR = os.path.dirname(os.path.dirname(hit[0]))
print("PART_DIR =", PART_DIR)

# her parça AYRI alt-klasör + binary maske -> nokta başına tek etikete birleştir
PART_SUBDIRS = sorted(d for d in glob.glob(os.path.join(PART_DIR,"points_label","*")) if os.path.isdir(d))
PART_NAMES = [os.path.basename(d) for d in PART_SUBDIRS]; NUM_PARTS = len(PART_SUBDIRS)
assert NUM_PARTS >= 2, f"parça alt-klasörü yok: {PART_DIR}/points_label/"
print("parçalar:", PART_NAMES, "| NUM_PARTS:", NUM_PARTS)

_pts = sorted(glob.glob(os.path.join(PART_DIR, "points", "*.pts")))
def _load(i):
    p = _pts[i]; mid = os.path.splitext(os.path.basename(p))[0]
    xyz = np.loadtxt(p, dtype=np.float32)[:, :3]; lab = np.zeros(len(xyz), np.int64)
    for pi, pd in enumerate(PART_SUBDIRS):
        sf = os.path.join(pd, mid + ".seg")
        if os.path.exists(sf):
            v = np.loadtxt(sf)
            if len(v) == len(xyz): lab[v != 0] = pi
    return xyz, lab

_rng = np.random.default_rng(0)
PALETTE = np.array([[.85,.20,.20],[.20,.65,.25],[.20,.35,.80],[.90,.75,.20],
                    [.60,.30,.75],[.25,.70,.70],[.90,.55,.20],[.5,.5,.5]])[:NUM_PARTS]

DATA = []
for j in range(min(N_MODELS, len(_pts))):
    xyz, part = _load(j)
    sidx = np.random.default_rng(j).choice(len(xyz), N_PTS, replace=len(xyz)<N_PTS)
    xyz, part = xyz[sidx], part[sidx]
    xyz = xyz - xyz.mean(0); xyz = xyz/(np.linalg.norm(xyz,axis=1).max()+1e-9)
    rgb = np.clip(PALETTE[part] + _rng.normal(0,CNOISE,(N_PTS,3)),0,1).astype(np.float32)
    gt = np.concatenate([xyz,rgb],1).astype(np.float32)
    partial, miss = separate_colored(gt, CROP, seed=j)
    DATA.append(dict(gt=gt, partial=partial, miss=miss, gt_part=part))
print(f"DATA hazır: {len(DATA)} model, {N_PTS} nokta, {NUM_PARTS} parça {PART_NAMES}")

---
## Senin yöntemin — 3 adım
1. **Geometriyi tamamla** — renksiz partial (xyz) → dondurulmuş **PoinTr** → tamamlanan xyz.
2. **Parçala** — renksiz tamamlanan bulut → **PointNet** part-seg → her noktanın parçası.
3. **Renklendir** — her nokta ← parçasının görünür-ortalama rengi → renkli tam bulut.

Baseline: NN-kopya. Ölçüt: eksik-bölge ΔE(Lab).

### STEP 1 · Dondurulmuş PoinTr (geometri)

In [ ]:
import torch
from easydict import EasyDict
from models.PoinTr import PoinTr, fps

DEV = "cuda"
cfg = EasyDict(trans_dim=384, knn_layer=1, num_pred=6144, num_query=96)
geo = PoinTr(cfg)
sd = torch.load(CKPT, map_location="cpu")
base = sd.get("base_model", sd.get("model", sd))
base = {k.replace("module.", ""): v for k, v in base.items()}
mi, ui = geo.load_state_dict(base, strict=False)
assert len(mi) == 0, f"orijinal PoinTr bekleniyordu, missing={mi[:4]} (0 olmalı)"
for p in geo.parameters(): p.requires_grad_(False)
geo.eval().to(DEV)
print(f"dondurulmuş orijinal PoinTr hazır | num_pred={geo.num_pred} | missing=0")

### STEP 1 · yardımcılar (complete_geometry + NN baseline)

In [ ]:
import numpy as np
from scipy.spatial import cKDTree

AXIS = [2, 1, 0]   # ShapeNet-Part -> PoinTr(ShapeNet-55) hizası: x,z swap (deneyle bulundu)

@torch.no_grad()
def complete_geometry(partial):
    """STEP 1: RENKSİZ partial (xyz) -> PoinTr tamamlaması. Doğru yönde ver, çıktıyı geri çevir."""
    x = partial[:, :3][:, AXIS]
    p = torch.from_numpy(x).float().unsqueeze(0).to(DEV)
    fine = geo(p)[1][0, :geo.num_pred].cpu().numpy()
    return fine[:, AXIS]                    # geri swap (DATA frame'ine dön)

def nn_color(partial, comp):               # BASELINE
    _, i = cKDTree(partial[:, :3]).query(comp[:, :3], k=1)
    return partial[i, 3:6]
print("STEP 1 hazır (complete_geometry x,z-swap + nn_color)")

### STEP 1 · görsel: PoinTr geometri tamamlama (renksiz)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

d = DATA[0]; partial = d["partial"]; gt = d["gt"]
comp = complete_geometry(partial)          # PoinTr: RENKSİZ geometri tamamlama (xyz)

def scat(xyz, color, size=1.8):
    return go.Scatter3d(x=xyz[:,0], y=xyz[:,1], z=xyz[:,2], mode="markers",
                        marker=dict(size=size, color=color))

fig = make_subplots(rows=1, cols=3, specs=[[{"type":"scene"}]*3],
                    subplot_titles=("occluded (girdi)", "PoinTr tamamlama (renksiz)", "GT (referans)"))
fig.add_trace(scat(partial[:,:3], "#8a8f96"), 1, 1)
fig.add_trace(scat(partial[:,:3], "#8a8f96"), 1, 2)
fig.add_trace(scat(comp, "#12a5b8"), 1, 2)
fig.add_trace(scat(gt[:,:3], "#8a8f96"), 1, 3)
for s in ("scene","scene2","scene3"): fig.layout[s].aspectmode = "data"
fig.update_layout(height=480, showlegend=False, margin=dict(l=0,r=0,t=30,b=0)); fig.show()

---
## STEP 2 · Part segmentation (PointNet)

### STEP 2 · PointNet part-seg modeli

In [ ]:
import torch, torch.nn as nn, numpy as np, glob, os

class _TNet(nn.Module):
    def __init__(s, k):
        super().__init__(); s.k = k
        s.mlp = nn.Sequential(nn.Conv1d(k,64,1),nn.BatchNorm1d(64),nn.ReLU(),
            nn.Conv1d(64,128,1),nn.BatchNorm1d(128),nn.ReLU(),
            nn.Conv1d(128,1024,1),nn.BatchNorm1d(1024),nn.ReLU())
        s.fc = nn.Sequential(nn.Linear(1024,512),nn.ReLU(),nn.Linear(512,256),nn.ReLU(),nn.Linear(256,k*k))
    def forward(s, x):
        B=x.size(0); f=s.mlp(x).max(-1)[0]
        return s.fc(f).view(B,s.k,s.k) + torch.eye(s.k,device=x.device).unsqueeze(0)

class PointNetPartSeg(nn.Module):
    def __init__(s, P):
        super().__init__(); s.itn=_TNet(3)
        s.mlp1=nn.Sequential(nn.Conv1d(3,64,1),nn.BatchNorm1d(64),nn.ReLU(),nn.Conv1d(64,128,1),nn.BatchNorm1d(128),nn.ReLU())
        s.fstn=_TNet(128)
        s.mlp2=nn.Sequential(nn.Conv1d(128,128,1),nn.BatchNorm1d(128),nn.ReLU(),nn.Conv1d(128,1024,1),nn.BatchNorm1d(1024),nn.ReLU())
        s.seg=nn.Sequential(nn.Conv1d(1152,512,1),nn.BatchNorm1d(512),nn.ReLU(),
            nn.Conv1d(512,256,1),nn.BatchNorm1d(256),nn.ReLU(),nn.Conv1d(256,P,1))
    def forward(s, x):                         # (B,N,3) -> (B,N,P)
        x=x.transpose(1,2); x=torch.bmm(s.itn(x),x)
        f=s.mlp1(x); f=torch.bmm(s.fstn(f),f); pf=f
        g=s.mlp2(f).max(-1,keepdim=True)[0].expand(-1,-1,f.size(-1))
        return s.seg(torch.cat([pf,g],1)).transpose(1,2)

def train_partseg(model, loader, epochs=30, lr=1e-3, device=DEV):
    model.to(device).train(); opt=torch.optim.Adam(model.parameters(),lr); lf=nn.CrossEntropyLoss()
    for ep in range(epochs):
        cor=seen=0; tot=0.0
        for xyz,lab in loader:
            xyz,lab=xyz.to(device),lab.to(device); opt.zero_grad()
            lo=model(xyz); loss=lf(lo.reshape(-1,lo.size(-1)),lab.reshape(-1))
            loss.backward(); opt.step()
            tot+=loss.item()*xyz.size(0); cor+=(lo.argmax(-1)==lab).sum().item(); seen+=lab.numel()
        if ep%5==0 or ep==epochs-1: print(f"  ep{ep:3d} loss {tot/len(loader.dataset):.4f} acc {cor/seen*100:.1f}%")
    return model

@torch.no_grad()
def segment(model, xyz, device=DEV):           # herhangi bir bulutu (PoinTr çıktısı dahil) etiketle
    model.eval(); x=np.asarray(xyz,np.float32)[:,:3]
    c=x.mean(0); x=(x-c)/(np.linalg.norm(x-c,axis=1).max()+1e-9)
    return model(torch.from_numpy(x).float().unsqueeze(0).to(device))[0].argmax(-1).cpu().numpy()
print("PointNet part-seg tanımlı")

### STEP 2 · PointNet'i eğit (renksiz GT + parça etiketi)

In [ ]:
# PointNet'i yukarıdaki renkli+etiketli GT bulutlarında eğit (ayrı ShapeNet-Part indirme yok)
from torch.utils.data import Dataset, DataLoader
class _GTPartDS(Dataset):
    def __init__(s, D): s.D = D
    def __len__(s): return len(s.D)
    def __getitem__(s, i):
        d = s.D[i]
        return torch.from_numpy(d["gt"][:, :3]).float(), torch.from_numpy(d["gt_part"]).long()

seg_model = PointNetPartSeg(NUM_PARTS).to(DEV)
train_partseg(seg_model, DataLoader(_GTPartDS(DATA), batch_size=16, shuffle=True), epochs=30)

### STEP 3 · Parçaya göre renklendir + ΔE + görsel (SENİN yöntemin)

In [ ]:
from scipy.spatial import cKDTree
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def part_color_pointnet(partial, comp, model):
    """STEP 3: tamamlanan (renksiz) bulutu PointNet'le parçala; her noktayı parçasının
       GÖRÜNÜR-ortalama rengiyle boya -> renkli tam bulut."""
    vl = segment(model, partial[:, :3]); cl = segment(model, comp[:, :3])
    P = int(max(vl.max(), cl.max())) + 1; vr = partial[:, 3:6]
    mean = np.tile(vr.mean(0), (P, 1))
    for k in range(P):
        m = vl == k
        if m.any(): mean[k] = vr[m].mean(0)
    return mean[cl]

# eksik-bölge ΔE: baseline (NN) vs SENİN yöntemin (PointNet-part)
nn_all, pn_all = [], []
for d in DATA[:20]:
    gt, partial = d["gt"], d["partial"]; comp = complete_geometry(partial)
    _, gi = cKDTree(gt[:, :3]).query(comp[:, :3], k=1)
    true_rgb = gt[gi, 3:6]; m = d["miss"][gi]; m = m if m.any() else np.ones(len(comp), bool)
    nn_all.append(deltaE(nn_color(partial, comp)[m], true_rgb[m]).mean())
    pn_all.append(deltaE(part_color_pointnet(partial, comp, seg_model)[m], true_rgb[m]).mean())
nn_all, pn_all = np.array(nn_all), np.array(pn_all)
print("eksik-bölge ΔE(Lab):")
print(f"  NN-kopya (baseline)  : {nn_all.mean():6.3f}")
print(f"  PointNet-part (senin): {pn_all.mean():6.3f}   (kazandığı: {(pn_all<nn_all).sum()}/{len(nn_all)})")

# görsel: occluded / NN / SENİN yöntemin / GT
d = DATA[0]; gt, partial = d["gt"], d["partial"]; comp = complete_geometry(partial)
nn_rgb = nn_color(partial, comp); pn_rgb = part_color_pointnet(partial, comp, seg_model)
def tr(a):
    c=["rgb(%d,%d,%d)"%(int(r*255),int(g*255),int(b*255)) for r,g,b in np.clip(a[:,3:6],0,1)]
    return go.Scatter3d(x=a[:,0],y=a[:,1],z=a[:,2],mode="markers",marker=dict(size=1.6,color=c))
panels=[("occluded",partial),("NN-kopya",np.vstack([partial,np.hstack([comp,nn_rgb])])),
        ("PointNet-part (senin)",np.vstack([partial,np.hstack([comp,pn_rgb])])),("GT",gt)]
fig=make_subplots(rows=1,cols=4,specs=[[{"type":"scene"}]*4],subplot_titles=[t for t,_ in panels])
for i,(_,a) in enumerate(panels,1): fig.add_trace(tr(a),1,i)
for s in ("scene","scene2","scene3","scene4"): fig.layout[s].aspectmode="data"
fig.update_layout(height=500,showlegend=False,margin=dict(l=0,r=0,t=30,b=0)); fig.show()